## Fundamental Concept: Risk-Neutral Pricing

Before introducing the pricing models, we start with one of the fundamental ideas in derivative pricing: **risk-neutral valuation**.

### What is risk-neutral pricing?

Under the **no-arbitrage assumption**, a derivative can be priced by constructing a self-financing portfolio composed of:

- the underlying asset, and
- a risk-free asset.

The portfolio is chosen so that it replicates the payoff of the derivative.

If the replicating portfolio and the derivative generate exactly the same future payoff, they must have the same value today. Otherwise, an arbitrage opportunity would exist: we could buy the cheaper position and sell the more expensive one while having no future risk.

Therefore, the price of the derivative does not depend directly on the real-world expected return of the underlying asset.

Instead, we can introduce a special probability measure called the **risk-neutral measure**, denoted by $Q$.

Under $Q$, the expected return of the underlying asset is the risk-free rate $r$.

For a derivative with payoff $X_T$ at maturity $T$, its value at time $t$ is:

$$
V_t
=
e^{-r(T-t)}
\mathbb{E}^Q
\left[
X_T \mid \mathcal{F}_t
\right].
$$

In particular, at time $t=0$:

$$
V_0
=
e^{-rT}
\mathbb{E}^Q[X_T].
$$

### Real-world probability vs. risk-neutral probability

In a binomial model, suppose the stock can move either up or down during each time step.

We can distinguish two probabilities:

- $p$: probability of an upward movement under the **real-world measure $P$**,
- $q$: probability of an upward movement under the **risk-neutral measure $Q$**.

In general:

$$
p \neq q.
$$

The real-world probability $p$ describes how we believe the stock actually behaves.

The risk-neutral probability $q$ is not necessarily a prediction of the real probability of an upward movement. It is the probability that makes the discounted asset price a martingale and allows us to price derivatives consistently with the no-arbitrage condition.

Under the risk-neutral measure:

$$
\mathbb{E}^Q[S_{t+\Delta t}\mid S_t]
=
S_t e^{r\Delta t}.
$$

This is why derivative pricing uses $Q$ rather than the real-world probability measure $P$.

## Cox-Ross-Rubinstein (CRR) Binomial Model

We split the interval $[0,T]$ into $N$ time steps:

$$
\Delta t = \frac{T}{N}.
$$

Parameters:

- $\sigma$: volatility,
- $T$: time to maturity,
- $r$: risk-free interest rate,
- $K$: strike price,
- $S_0$: initial stock price.

The CRR up and down factors are:

$$
u = e^{\sigma\sqrt{\Delta t}},
$$

$$
d = e^{-\sigma\sqrt{\Delta t}}.
$$

The risk-neutral probability of an upward movement is:

$$
p
=
\frac{e^{r\Delta t}-d}{u-d}.
$$

At node $(i,j)$ of the recombining tree, where $j$ is the time step and $i$ is the number of downward movements, the stock price is:

$$
S_{i,j}
=
S_0
u^{\,j-i}
d^{\,i}.
$$

Under the risk-neutral measure, the price of a European option is:

$$
V_0
=
e^{-rT}
\mathbb{E}^Q
\left[
\text{Payoff}(S_T)
\right].
$$

In the binomial tree, we compute option values backward using:

$$
V
=
e^{-r\Delta t}
\left[
pV_{\text{up}}
+
(1-p)V_{\text{down}}
\right].
$$

For an American option, at each node we compare:

$$
\text{Continuation Value}
$$

with the immediate exercise value:

$$
\text{Payoff}(S_{i,j}).
$$

Therefore:

$$
V_{i,j}
=
\max
\left(
\text{Payoff}(S_{i,j}),
e^{-r\Delta t}
\left[
pV_{\text{up}}
+
(1-p)V_{\text{down}}
\right]
\right).
$$

If the immediate payoff is larger than the continuation value, it is optimal to exercise the option early.

As $N\to\infty$, the CRR price of a European option converges to the Black-Scholes price.

In [2]:
from math import sqrt , exp
import numpy as np

In [3]:
#lets craft the binomial Tree

def BTree(sigma:float,r:float,T:float,N:int,S0:float):
    dt=T/N
    u=exp(sigma*sqrt(dt)) #up coef
    d=exp(-sigma*sqrt(dt)) #down coef
    p=(exp(r*dt)-d)/(u-d) #risk netural probability
    S=np.zeros((N+1,N+1))
    S[0,0]=S0
    z=1
    for j in range (1,N+1):
        for i in range (z):
            #when its up and when its down
            S[i,j]=S[i,j-1]*u
            S[i+1,j]=S[i,j-1]*d
        z+=1            
    
    return S,p
    
def Payoff(K:float,Spot:np.array,OptionType:chr):
    if OptionType=="call":
        return np.maximum(Spot-K,0)
    return np.maximum(K-Spot,0)
    
    
def PriceEurope(sigma:float,r:float,T:float,N:int,S0:float,K:float,OptionType:chr):
    Spots,p=BTree(sigma,r,T,N,S0)
    dt=T/N
    V=np.zeros((N+1,N+1)) #V[i,N] we need to fill this last column

        #first we fill in the N+1 column with the payoffs
    for i in range (N+1):
        V[i,N]=Payoff(K,Spots[i,N],OptionType)
            
    for j in range (N-1,-1,-1): 
        for i in range (j+1): #backward induction each time we remove a row and calculate V
            V[i,j]=exp(-r*dt)*(p*V[i,j+1]+(1-p)*V[i+1,j+1])
    return V

def PriceAmerican(sigma:float,r:float,T:float,N:int,S0:float,K:float,OptionType:chr):
    spots,p=BTree(sigma,r,T,N,S0)
    dt=T/N
    V=np.zeros((N+1,N+1))
    for i in range(N+1):
        V[i,N]=Payoff(K,spots[i,N],OptionType)
    for j in range(N-1,-1,-1):
        for i in range(j+1):
            V[i,j]=np.maximum(Payoff(K,spots[i,j],OptionType),exp(-r*dt)*(p*V[i,j+1]+(1-p)*V[i+1,j+1]))
    return V



In [4]:
#testing:
S0=36
T=1.0
sigma=0.2
r=0.06
N=1000
K=35

In [5]:
print(PriceEurope(sigma,r,T,N,S0,K,"call")[0,0])

4.527498368435431


In [6]:
print(PriceAmerican(sigma,r,T,N,S0,K,"put")[0,0])

1.6578184137816894


## Black-Scholes Model

Under the risk-neutral measure $Q$, the stock price follows the stochastic differential equation:

$$
dS_t = rS_t\,dt + \sigma S_t\,dW_t^Q
$$

where $W_t^Q$ is a Brownian motion under the risk-neutral measure.

The solution of this Geometric Brownian Motion between time $t$ and maturity $T$ is:

$$
S_T
=
S_t
\exp\left[
\left(r-\frac{\sigma^2}{2}\right)(T-t)
+
\sigma\sqrt{T-t}\,Z
\right],
\qquad
Z\sim\mathcal{N}(0,1).
$$

Under risk-neutral valuation, the value of a derivative with payoff $f(S_T)$ is:

$$
V_t
=
e^{-r(T-t)}
\mathbb{E}^Q
\left[
f(S_T)\mid\mathcal{F}_t
\right].
$$

For a European call option, the payoff at maturity is:

$$
f(S_T)=(S_T-K)^+.
$$

Using the Black-Scholes formula, the call price at time $t$ is:

$$
C_t
=
S_tN(d_1)
-
Ke^{-r(T-t)}N(d_2),
$$

where:

$$
d_1
=
\frac{
\ln\left(\frac{S_t}{K}\right)
+
\left(r+\frac{\sigma^2}{2}\right)(T-t)
}{
\sigma\sqrt{T-t}
},
$$

and:

$$
d_2
=
\frac{
\ln\left(\frac{S_t}{K}\right)
+
\left(r-\frac{\sigma^2}{2}\right)(T-t)
}{
\sigma\sqrt{T-t}
}
=
d_1-\sigma\sqrt{T-t}.
$$

Here, $N(x)$ is the cumulative distribution function of a standard normal random variable:

$$
N(x)
=
\frac{1}{\sqrt{2\pi}}
\int_{-\infty}^{x}
e^{-u^2/2}\,du.
$$

In [7]:
import scipy.integrate as integrate

def N(u):
        return (integrate.quad(lambda x :(1/np.sqrt(2*np.pi))*exp((-x**2)/2),-np.inf,u))[0]
    
def BlackScholes(sigma,r,T,K,S,OptionType):
    d1=(np.log(S/K)+(r+(sigma**2)/2)*T)/(sigma*sqrt(T))
    d2=(np.log(S/K)+(r-(sigma**2)/2)*T)/(sigma*sqrt(T))
    
    if OptionType=="call":
        V=exp(-r*T)*(exp(r*T)*S*N(d1)-K*N(d2))
    if OptionType=="put":
        V=exp(-r*T)*(K*N(-d2)-exp(r*T)*S*N(-d1))
    return V
    

In [8]:
print(BlackScholes(sigma,r,T,K,S0,"call"))

4.527296534725704


## Monte Carlo Simulation

We will use another method, **Monte Carlo simulation**, to price derivatives.

Under the risk-neutral measure $Q$, we assume that the stock price $S_t$ follows a Geometric Brownian Motion (GBM):

$$
dS_t = rS_t\,dt + \sigma S_t\,dW_t^Q
$$

where:

- $r$ is the risk-free interest rate,
- $\sigma$ is the volatility,
- $W_t^Q$ is a Brownian motion under the risk-neutral measure.

### Discretization of Brownian motion

We divide the interval $[0,T]$ into $N$ steps:

$$
\Delta t = \frac{T}{N}
$$

Brownian increments are independent and satisfy:

$$
W_{t+\Delta t} - W_t
\sim
\mathcal{N}(0,\Delta t)
$$

Therefore, we can write:

$$
\Delta W_t
=
\sqrt{\Delta t}\,Z,
\qquad
Z \sim \mathcal{N}(0,1)
$$

### Euler-Maruyama method

Using the Euler approximation:

$$
S_{t+\Delta t}
=
S_t
+
rS_t\Delta t
+
\sigma S_t\sqrt{\Delta t}\,Z
$$

Equivalently:

$$
S_{t+\Delta t}
=
S_t
\left(
1+r\Delta t+\sigma\sqrt{\Delta t}\,Z
\right)
$$

### Exact simulation of GBM

Since GBM has a known analytical solution:

$$
S_t
=
S_0
\exp\left[
\left(r-\frac{\sigma^2}{2}\right)t
+
\sigma\sqrt{t}\,Z
\right]
$$

where:

$$
Z \sim \mathcal{N}(0,1)
$$

In [22]:
import matplotlib.pyplot as plt

def GBM(S0,r,sigma,T,N): 
    dt=T/N
    S=np.zeros(N+1)
    S[0]=S0
    for i in range(1, N + 1):
        Z = np.random.normal(0, 1)

        S[i] = S[i-1] * np.exp(
            (r - sigma**2 / 2) * dt
            + sigma * np.sqrt(dt) * Z)     
    return S
def MCS(S0,r,sigma,T,N,OptionType,npaths,K):
    P=0
    for i in range (npaths):
        S=GBM(S0,r,sigma,T,N)
        P+=Payoff(K,S[N],OptionType)
    return exp(-r*T)*P/npaths

npaths=1000
N=1000
OptionType="call"
print(MCS(S0,r,sigma,T,N,OptionType,npaths,K))

4.786201342868826
